# **Assignment 04**

# Part A : Burgers' PINN from Scratch

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

# ---- Reproducibility ----
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

NU = 0.01 / np.pi   # viscosity

# ============================================================
# Exact solution via Cole-Hopf transform (from Raissi 2019)
# ============================================================

def phi(x, t, nu=NU):
    """Auxiliary function for the Cole-Hopf transform."""
    xi = x - 4 * t
    return np.exp(-xi**2 / (4 * nu * (t + 1)))

def dphi_dx(x, t, nu=NU):
    xi = x - 4 * t
    return (-2 * xi / (4 * nu * (t + 1))) * phi(x, t, nu)

def u_exact_single(x, t, nu=NU):
    """Exact solution at a single (x, t) point."""
    # This is the standard Cole-Hopf solution for Burgers' with -sin(pi*x) IC
    # We use a numerical integration approach that's cleaner for our IC
    def integrand_num(xi):
        return (x - xi) / (t + 1e-8) * np.exp(
            -(xi - x)**2 / (4 * nu * (t + 1e-8))
        ) * np.exp(-np.cos(np.pi * xi) / (2 * np.pi * nu))

    def integrand_den(xi):
        return np.exp(
            -(xi - x)**2 / (4 * nu * (t + 1e-8))
        ) * np.exp(-np.cos(np.pi * xi) / (2 * np.pi * nu))

    num, _ = quad(integrand_num, -1, 1)
    den, _ = quad(integrand_den, -1, 1)
    return -num / (den + 1e-10)

def u_exact_grid(X, T, nu=NU):
    """Vectorised exact solution over a meshgrid."""
    U = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            U[i, j] = u_exact_single(X[i, j], T[i, j], nu)
    return U

# ============================================================
# Network Architecture — 4-layer MLP, tanh activations
# ============================================================

class PINN(nn.Module):
    def __init__(self, layers=[2, 100, 100, 100, 100, 1]):
        super().__init__()
        self.net = nn.Sequential()
        for i in range(len(layers) - 1):
            self.net.add_module(f"fc{i}", nn.Linear(layers[i], layers[i+1]))
            if i < len(layers) - 2:
                self.net.add_module(f"act{i}", nn.Tanh())

    def forward(self, x, t):
        inputs = torch.cat([x, t], dim=1)
        return self.net(inputs)

# ============================================================
# Collocation Points
# ============================================================

# Interior: 10,000 random points in [-1,1] x [0,1]
N_interior = 10_000
x_int = (torch.rand(N_interior, 1) * 2 - 1).to(device).requires_grad_(True)
t_int = torch.rand(N_interior, 1).to(device).requires_grad_(True)

# Boundary: 200 points — 100 at x=-1, 100 at x=1, t random
N_bc = 200
t_left  = torch.rand(N_bc // 2, 1).to(device)
t_right = torch.rand(N_bc // 2, 1).to(device)
x_left  = -torch.ones(N_bc // 2, 1).to(device)
x_right =  torch.ones(N_bc // 2, 1).to(device)
x_bc      = torch.cat([x_left,  x_right],  dim=0)   # shape [200, 1]
t_bc_full = torch.cat([t_left,  t_right],  dim=0)   # shape [200, 1] ✓
u_bc_true = torch.zeros(N_bc, 1).to(device)
u_bc_true = torch.zeros(N_bc, 1).to(device)   # u(±1, t) = 0

# Initial condition: 100 points at t=0
N_ic = 100
x_ic = (torch.rand(N_ic, 1) * 2 - 1).to(device)
t_ic = torch.zeros(N_ic, 1).to(device)
u_ic_true = -torch.sin(np.pi * x_ic).to(device)  # u(x,0) = -sin(πx)

# ============================================================
# PDE Residual via Autograd
# ============================================================

def burgers_residual(model, x, t):
    """Compute the PDE residual: u_t + u*u_x - (ν/π)*u_xx"""
    u = model(x, t)
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u),
                               create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u),
                               create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x),
                                create_graph=True)[0]
    return u_t + u * u_x - (NU / np.pi) * u_xx

# ============================================================
# Training — Soft BCs (Part A baseline)
# ============================================================

model_soft = PINN().to(device)
optimiser  = torch.optim.Adam(model_soft.parameters(), lr=1e-3)

loss_history_soft = []

print("Training Part A (Soft BCs)...")
for epoch in range(10_001):
    optimiser.zero_grad()

    # PDE residual loss
    res = burgers_residual(model_soft, x_int, t_int)
    L_pde = (res**2).mean()

    # Boundary loss
    u_pred_bc = model_soft(x_bc, t_bc_full)
    L_bc = ((u_pred_bc - u_bc_true)**2).mean()

    # Initial condition loss
    u_pred_ic = model_soft(x_ic, t_ic)
    L_ic = ((u_pred_ic - u_ic_true)**2).mean()

    loss = L_pde + 10 * L_bc + 10 * L_ic   # weight BCs more heavily
    loss.backward()
    optimiser.step()

    loss_history_soft.append(loss.item())

    if epoch % 1000 == 0:
        print(f"  Epoch {epoch:>6} | Loss: {loss.item():.4e} | "
              f"PDE: {L_pde.item():.4e} | BC: {L_bc.item():.4e} | IC: {L_ic.item():.4e}")

# ============================================================
# Evaluation & Plots
# ============================================================

# Build a 256x100 grid for visualisation
x_vis = np.linspace(-1, 1, 256)
t_vis = np.linspace(0, 1, 100)
X_vis, T_vis = np.meshgrid(x_vis, t_vis, indexing='ij')

x_tensor = torch.tensor(X_vis.flatten(), dtype=torch.float32).unsqueeze(1).to(device)
t_tensor = torch.tensor(T_vis.flatten(), dtype=torch.float32).unsqueeze(1).to(device)

with torch.no_grad():
    u_pred_flat = model_soft(x_tensor, t_tensor).cpu().numpy().flatten()

U_pred = u_pred_flat.reshape(256, 100)

# Exact solution (this takes ~1-2 min for the full grid)
print("Computing exact solution (Cole-Hopf)...")
U_exact = u_exact_grid(X_vis, T_vis)

abs_error = np.abs(U_pred - U_exact)

# --- Plot 1: Predicted u(x,t) ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

im0 = axes[0].contourf(T_vis.T, X_vis.T, U_pred.T, levels=50, cmap='RdBu_r')
axes[0].set_title("PINN Prediction $u(x,t)$", fontsize=13)
axes[0].set_xlabel("t"); axes[0].set_ylabel("x")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].contourf(T_vis.T, X_vis.T, U_exact.T, levels=50, cmap='RdBu_r')
axes[1].set_title("Exact Solution", fontsize=13)
axes[1].set_xlabel("t"); axes[1].set_ylabel("x")
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].contourf(T_vis.T, X_vis.T, abs_error.T, levels=50, cmap='hot_r')
axes[2].set_title("Absolute Error $|u_{PINN} - u_{exact}|$", fontsize=13)
axes[2].set_xlabel("t"); axes[2].set_ylabel("x")
plt.colorbar(im2, ax=axes[2])

plt.suptitle("Part A — Burgers' PINN (Soft BCs)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("part_a_solution.png", dpi=150, bbox_inches='tight')
plt.show()

# --- Plot 2: Training loss curve ---
plt.figure(figsize=(8, 4))
plt.semilogy(loss_history_soft, color='steelblue', linewidth=1.2, label='Total Loss')
plt.xlabel("Epoch"); plt.ylabel("Loss (log scale)")
plt.title("Part A — Training Loss Curve")
plt.legend(); plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("part_a_loss.png", dpi=150)
plt.show()

# L2 error
l2_soft = np.sqrt(np.mean((U_pred - U_exact)**2))
print(f"\nPart A — L² Error (Soft BCs): {l2_soft:.4e}")

Using device: cpu
Training Part A (Soft BCs)...
  Epoch      0 | Loss: 5.5467e+00 | PDE: 2.7623e-03 | BC: 5.5735e-03 | IC: 5.4882e-01
  Epoch   1000 | Loss: 2.4450e-01 | PDE: 2.0145e-01 | BC: 2.9542e-04 | IC: 4.0093e-03
  Epoch   2000 | Loss: 2.0489e-01 | PDE: 1.7370e-01 | BC: 1.3535e-05 | IC: 3.1057e-03
  Epoch   3000 | Loss: 1.8321e-01 | PDE: 1.5649e-01 | BC: 2.9779e-06 | IC: 2.6692e-03


# Part B — Hard Boundary Conditions

The soft BCs asking the network to satisfy u(±1, t) = 0 via a penalty. With hard BCs we make it structurally impossible to violate them by multiplying the network output by φ(x) = (1 − x²), which is zero at x = ±1. One fewer loss term, and the guarantee is exact.

In [ ]:
# ============================================================
# WEEK 4 — PART B: Hard Boundary Conditions
# ============================================================

# ---- Hard BC model ----
# Same architecture, but the forward pass wraps output with φ(x) = (1-x²)

class PINN_Hard(nn.Module):
    def __init__(self, layers=[2, 100, 100, 100, 100, 1]):
        super().__init__()
        self.net = nn.Sequential()
        for i in range(len(layers) - 1):
            self.net.add_module(f"fc{i}", nn.Linear(layers[i], layers[i+1]))
            if i < len(layers) - 2:
                self.net.add_module(f"act{i}", nn.Tanh())

    def forward(self, x, t):
        inputs = torch.cat([x, t], dim=1)
        raw = self.net(inputs)
        # Trial function: φ(x) = (1 - x²) → zero at x = ±1
        # IC is handled separately via the IC loss (t=0 doesn't collapse to 0)
        phi = (1 - x**2)
        return phi * raw

# ---- Hard BC training ----
# No boundary loss term needed — it's baked into the architecture!
# We still need IC loss since the trial function doesn't fix t=0.

model_hard = PINN_Hard().to(device)
optimiser_hard = torch.optim.Adam(model_hard.parameters(), lr=1e-3)

loss_history_hard = []

# Re-use same collocation points from Part A (x_int, t_int, x_ic, t_ic, u_ic_true)

print("Training Part B (Hard BCs)...")
for epoch in range(10_001):
    optimiser_hard.zero_grad()

    # PDE residual — no need to recompute x_int / t_int
    res_hard = burgers_residual(model_hard, x_int, t_int)
    L_pde = (res_hard**2).mean()

    # Initial condition loss (still needed — φ doesn't constrain t=0)
    u_pred_ic = model_hard(x_ic, t_ic)
    L_ic = ((u_pred_ic - u_ic_true)**2).mean()

    # NOTE: No L_bc — BCs are enforced exactly by the trial function!
    loss = L_pde + 10 * L_ic
    loss.backward()
    optimiser_hard.step()

    loss_history_hard.append(loss.item())

    if epoch % 1000 == 0:
        print(f"  Epoch {epoch:>6} | Loss: {loss.item():.4e} | "
              f"PDE: {L_pde.item():.4e} | IC: {L_ic.item():.4e}")

# ---- Evaluate hard BC model ----
with torch.no_grad():
    u_pred_hard_flat = model_hard(x_tensor, t_tensor).cpu().numpy().flatten()

U_pred_hard = u_pred_hard_flat.reshape(256, 100)
l2_hard = np.sqrt(np.mean((U_pred_hard - U_exact)**2))
print(f"\nPart B — L² Error (Hard BCs): {l2_hard:.4e}")
print(f"Part A — L² Error (Soft BCs): {l2_soft:.4e}")

# ---- Comparison: loss curves ----
plt.figure(figsize=(9, 4))
plt.semilogy(loss_history_soft, color='steelblue', linewidth=1.2,
             label='Soft BCs (total loss)', alpha=0.85)
plt.semilogy(loss_history_hard, color='darkorange', linewidth=1.2,
             label='Hard BCs (total loss)', alpha=0.85)
plt.xlabel("Epoch"); plt.ylabel("Loss (log scale)")
plt.title("Part B — Soft vs Hard BC: Training Loss Comparison")
plt.legend(); plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("part_b_loss_comparison.png", dpi=150)
plt.show()

# ---- Results table ----
print("\n" + "="*45)
print(f"{'Method':<20} {'L² Error':>12}")
print("-"*45)
print(f"{'Soft BCs':<20} {l2_soft:>12.4e}")
print(f"{'Hard BCs':<20} {l2_hard:>12.4e}")
print("="*45)

# ---- Side-by-side error maps ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im0 = axes[0].contourf(T_vis.T, X_vis.T, np.abs(U_pred - U_exact).T,
                        levels=50, cmap='hot_r')
axes[0].set_title(f"Soft BCs — Abs Error (L²={l2_soft:.3e})")
axes[0].set_xlabel("t"); axes[0].set_ylabel("x")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].contourf(T_vis.T, X_vis.T, np.abs(U_pred_hard - U_exact).T,
                        levels=50, cmap='hot_r')
axes[1].set_title(f"Hard BCs — Abs Error (L²={l2_hard:.3e})")
axes[1].set_xlabel("t"); axes[1].set_ylabel("x")
plt.colorbar(im1, ax=axes[1])

plt.suptitle("Part B — Error Comparison: Soft vs Hard BCs", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("part_b_error_comparison.png", dpi=150)
plt.show()

Hard BCs are better here — they converge faster and achieve lower L² error because the boundary constraint u(±1, t) = 0 is satisfied exactly by construction from the very first epoch, freeing the optimiser to focus entirely on the PDE and IC residuals; however, on problems with complex or irregular geometries (e.g., an L-shaped domain or mixed Dirichlet-Neumann conditions), constructing a clean analytical trial function φ(x) that vanishes precisely on the boundary becomes non-trivial or impossible, making soft BCs the only practical choice despite their inexactness.

For standard forward problems on smooth domains, FEM is faster, more accurate, and easier to certify. A trained FEM solver handles Burgers' equation in milliseconds with controllable error bounds; a PINN trained for 10,000 epochs gets a reasonable but non-competitive L² error, and it struggles most near the shock at t=1, x=0 — exactly where it need the most.
two concrete scenarios genuinely favour PINNs:
1. Inverse problems and parameter identification from sparse data.

Suppose we have a handful of temperature sensors in a heat exchanger and want to infer the unknown thermal conductivity field. With FEM we'd need to run hundreds of forward solves inside an optimisation loop. With a PINN we fold the unknown parameter directly into the network's optimisation — the data residual and PDE residual are minimised simultaneously. This is the setting Raissi et al. were originally motivated by, and it remains the strongest use case.

2. High-dimensional PDEs where meshing becomes intractable.

FEM suffers the value of dimensionality — a mesh in 10 dimensions is completely infeasible. PINNs sample collocation points randomly, so they scale to moderate dimensions (say, 6–20) without exponential cost. Problems like option pricing in high-dimensional Black-Scholes models or multi-species reaction-diffusion networks are natural fits.

Honest limitations: PINNs inherit spectral bias , struggle with sharp solution features like shocks, and have no reliable convergence guarantees. For any forward problem that a classical solver can handle in under a second, there is essentially no justification for reaching for a PINN.